In [ ]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fedesoriano/cifar10-python-in-csv")

print("Path to dataset files:", path)

100%|██████████| 209M/209M [00:02<00:00, 85.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/fedesoriano/cifar10-python-in-csv/versions/1


In [ ]:
import os

print("Dataset path:", path)

for root, dirs, files in os.walk(path):
    print(root, "->", len(files), "files")
    if root.count(os.sep) - path.count(os.sep) >= 2:
        dirs[:] = []

Dataset path: /root/.cache/kagglehub/datasets/fedesoriano/cifar10-python-in-csv/versions/1
/root/.cache/kagglehub/datasets/fedesoriano/cifar10-python-in-csv/versions/1 -> 3 files


In [ ]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [ ]:
os.listdir(path)

['test.csv', 'batches.meta', 'train.csv']

In [ ]:
import os
import pandas as pd

print(os.listdir(path))

train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

print(train_df.shape)
print(train_df.head())
print(test_df.shape)
print(test_df.head())

['test.csv', 'batches.meta', 'train.csv']
(50000, 3073)
   pixel_0  pixel_1  pixel_2  pixel_3  pixel_4  pixel_5  pixel_6  pixel_7  \
0       59       43       50       68       98      119      139      145   
1      154      126      105      102      125      155      172      180   
2      255      253      253      253      253      253      253      253   
3       28       37       38       42       44       40       40       24   
4      170      168      177      183      181      177      181      184   

   pixel_8  pixel_9  ...  pixel_3063  pixel_3064  pixel_3065  pixel_3066  \
0      149      149  ...          58          65          59          46   
1      142      111  ...          42          67         101         122   
2      253      253  ...          83          80          69          66   
3       32       43  ...          39          59          42          44   
4      189      189  ...          88          85          82          83   

   pixel_3067  pixel_306

In [ ]:
print("Train shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())
print(train_df.head())
print("\nTest shape:", test_df.shape)
print(test_df.head())


Train shape: (50000, 3073)
Columns: ['pixel_0', 'pixel_1', 'pixel_2', 'pixel_3', 'pixel_4', 'pixel_5', 'pixel_6', 'pixel_7', 'pixel_8', 'pixel_9', 'pixel_10', 'pixel_11', 'pixel_12', 'pixel_13', 'pixel_14', 'pixel_15', 'pixel_16', 'pixel_17', 'pixel_18', 'pixel_19', 'pixel_20', 'pixel_21', 'pixel_22', 'pixel_23', 'pixel_24', 'pixel_25', 'pixel_26', 'pixel_27', 'pixel_28', 'pixel_29', 'pixel_30', 'pixel_31', 'pixel_32', 'pixel_33', 'pixel_34', 'pixel_35', 'pixel_36', 'pixel_37', 'pixel_38', 'pixel_39', 'pixel_40', 'pixel_41', 'pixel_42', 'pixel_43', 'pixel_44', 'pixel_45', 'pixel_46', 'pixel_47', 'pixel_48', 'pixel_49', 'pixel_50', 'pixel_51', 'pixel_52', 'pixel_53', 'pixel_54', 'pixel_55', 'pixel_56', 'pixel_57', 'pixel_58', 'pixel_59', 'pixel_60', 'pixel_61', 'pixel_62', 'pixel_63', 'pixel_64', 'pixel_65', 'pixel_66', 'pixel_67', 'pixel_68', 'pixel_69', 'pixel_70', 'pixel_71', 'pixel_72', 'pixel_73', 'pixel_74', 'pixel_75', 'pixel_76', 'pixel_77', 'pixel_78', 'pixel_79', 'pixel_80', '

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Separate pixels and labels
X_train = train_df.drop(columns=["label"]).values if "label" in train_df.columns else train_df.iloc[:, :-1].values
y_train = train_df["label"].values

# Convert pixels into RGB images: (N, 32, 32, 3)
X_train = X_train.reshape(-1, 3, 32, 32).astype(np.float32) / 255.0

# Normalize to [-1, 1]
X_train = (X_train - 0.5) / 0.5

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)

# Encode string labels as integers
classes = sorted(np.unique(y_train))
class_to_idx = {name: i for i, name in enumerate(classes)}
y_train = torch.tensor(
    [class_to_idx[label] for label in y_train],
    dtype=torch.long
)

# Create training dataset and loader
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

print("Images:", X_train.shape)
print("Labels:", y_train.shape)
print("Classes:", classes)

Images: torch.Size([50000, 3, 32, 32])
Labels: torch.Size([50000])
Classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [ ]:
image,lable=train_data[0]


In [ ]:
image.size()

torch.Size([3, 32, 32])

In [ ]:
class_names=['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']


In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet,self).__init__()
        self.conv1=nn.Conv2d(3,12,5)
        self.pool=nn.MaxPool2d(2,2)
        self.conv2=nn.Conv2d(12,24,5)
        self.fc1=nn.Linear(24*5*5,120)
        self.fc2=nn.Linear(120,84)
        self.fc3=nn.Linear(84,10)

    def forward(self,x):
        x=self.pool(f.relu(self.conv1(x)))
        x=self.pool(f.relu(self.conv2(x)))
        x=torch.flatten(x,1)
        x=f.relu(self.fc1(x))
        x=f.relu(self.fc2(x))
        x=self.fc3(x)
        return x





In [ ]:
net=NeuralNet()
loss_function=nn.CrossEntropyLoss()
optimizer=optim.SGD(net.parameters(),lr=0.001,momentum=0.9)


In [ ]:
for epoch in range(40):

    print(f'Epoch {epoch} ..... ')
    running_loss=0.0
    for i,data in enumerate(train_loader):
        inputs,labels=data
        optimizer.zero_grad()
        outputs=net(inputs)
        loss=loss_function(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'{running_loss/len(train_loader):.4f}')
print('Finished Training')

Epoch 0 ..... 
1.7860
Epoch 1 ..... 
1.6032
Epoch 2 ..... 
1.4935
Epoch 3 ..... 
1.4187
Epoch 4 ..... 
1.3636
Epoch 5 ..... 
1.3086
Epoch 6 ..... 
1.2594
Epoch 7 ..... 
1.2125
Epoch 8 ..... 
1.1725
Epoch 9 ..... 
1.1297
Epoch 10 ..... 
1.0962
Epoch 11 ..... 
1.0629
Epoch 12 ..... 
1.0366
Epoch 13 ..... 
1.0090
Epoch 14 ..... 
0.9836
Epoch 15 ..... 
0.9617
Epoch 16 ..... 
0.9390
Epoch 17 ..... 
0.9119
Epoch 18 ..... 
0.8917
Epoch 19 ..... 
0.8726
Epoch 20 ..... 
0.8529
Epoch 21 ..... 
0.8312
Epoch 22 ..... 
0.8150
Epoch 23 ..... 
0.7984
Epoch 24 ..... 
0.7802
Epoch 25 ..... 
0.7608
Epoch 26 ..... 
0.7432
Epoch 27 ..... 
0.7283
Epoch 28 ..... 
0.7113
Epoch 29 ..... 
0.6939
Epoch 30 ..... 
0.6818
Epoch 31 ..... 
0.6631
Epoch 32 ..... 
0.6488
Epoch 33 ..... 
0.6333
Epoch 34 ..... 
0.6194
Epoch 35 ..... 
0.6069
Epoch 36 ..... 
0.5901
Epoch 37 ..... 
0.5742
Epoch 38 ..... 
0.5592
Epoch 39 ..... 
0.5487
Finished Training



*    plane->airplane   
*   car->automobile
*   bird
*   cat
*   deer
*   dog
*   frog
*   horse
*   ship
*   truck

In [ ]:
torch.save(net.state_dict(),'./cifar_net.pth')


In [ ]:
net=NeuralNet()
net.load_state_dict(torch.load('./cifar_net.pth'))

<All keys matched successfully>

In [ ]:
correct=0
total=0
net.eval()

with torch.no_grad():
    for data in train_loader:
        images,labels=data
        outputs=net(images)
        _,predicted=torch.max(outputs,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
accuracy=100*correct/total


Accuracy: {accuracy}%


In [ ]:
print(accuracy)

82.304


In [ ]:
new_transform=transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])

def load_image(img_path):
    image=Image.open(img_path)
    image=new_transform(image)
    image=torch.unsqueeze(image,0)
    return image


image_path=['/content/data/deer_image.jfif','/content/data/horse_image.jfif','/content/data/plane_image.jfif','/content/data/dog_image.jfif','/content/data/cat_image.jfif','/content/data/frog_image.jfif','/content/data/truck_image.jfif']
images=[load_image(img) for img in image_path]
net.eval()
with torch.no_grad():
    for image in images:
        outputs=net(image)
        _,predicted=torch.max(outputs,1)
        print(f"Prediction: {class_names[predicted.item()]}, {predicted}")



Prediction: deer, tensor([4])
Prediction: horse, tensor([7])
Prediction: plane, tensor([0])
Prediction: horse, tensor([7])
Prediction: dog, tensor([5])
Prediction: frog, tensor([6])
Prediction: truck, tensor([9])


In [ ]:
!git clone https://github.com/abhiramthota2210/5.CIFAR-10-cnn-classification-colab

Cloning into '5.CIFAR-10-cnn-classification-colab'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [62]:
!cp "/content/CIFAR_10CNN.ipynb" "/content/5.CIFAR-10-cnn-classification-colab/"

cp: cannot stat '/content/CIFAR_10CNN.ipynb': No such file or directory
